In [ ]:
import numpy as np

In [ ]:
base_path = '../mma/livecell model performance'

model = 'DINOCell'
dinocell_mmas = np.load(f'{base_path}/{model}/mmas.npy')
dinocell_mma_greedys = np.load(f'{base_path}/{model}/mma_greedys.npy')
dinocell_ajis = np.load(f'{base_path}/{model}/ajis.npy')
dinocell_segs = np.load(f'{base_path}/{model}/segs.npy')
dinocell_pqs = np.load(f'{base_path}/{model}/pqs.npy')
dinocell_ap_50s = np.load(f'{base_path}/{model}/ap_50s.npy')

model = 'Cellpose-SAM'
cellpose_sam_mmas = np.load(f'{base_path}/{model}/mmas.npy')
cellpose_sam_mma_greedys = np.load(f'{base_path}/{model}/mma_greedys.npy')
cellpose_sam_ajis = np.load(f'{base_path}/{model}/ajis.npy')
cellpose_sam_segs = np.load(f'{base_path}/{model}/segs.npy')
cellpose_sam_pqs = np.load(f'{base_path}/{model}/pqs.npy')
cellpose_sam_ap_50s = np.load(f'{base_path}/{model}/ap_50s.npy')

model = 'SAMCell-LiveCell'
samcell_livecell_mmas = np.load(f'{base_path}/{model}/mmas.npy')
samcell_livecell_mma_greedys = np.load(f'{base_path}/{model}/mma_greedys.npy')
samcell_livecell_ajis = np.load(f'{base_path}/{model}/ajis.npy')
samcell_livecell_segs = np.load(f'{base_path}/{model}/segs.npy')
samcell_livecell_pqs = np.load(f'{base_path}/{model}/pqs.npy')
samcell_livecell_ap_50s = np.load(f'{base_path}/{model}/ap_50s.npy')

model = 'SAMCell-cyto'
samcell_cyto_mmas = np.load(f'{base_path}/{model}/mmas.npy')
samcell_cyto_mma_greedys = np.load(f'{base_path}/{model}/mma_greedys.npy')
samcell_cyto_ajis = np.load(f'{base_path}/{model}/ajis.npy')
samcell_cyto_segs = np.load(f'{base_path}/{model}/segs.npy')
samcell_cyto_pqs = np.load(f'{base_path}/{model}/pqs.npy')
samcell_cyto_ap_50s = np.load(f'{base_path}/{model}/ap_50s.npy')

In [ ]:
def compute_rankings(dinocell_scores, cellpose_sam_scores, samcell_livecell_scores, samcell_cyto_scores):
    # merged_scores = np.concat((np.reshape(dinocell_scores, (-1, 1)), np.reshape(cellpose_sam_scores, (-1, 1)), np.reshape(samcell_livecell_scores, (-1, 1)), np.reshape(samcell_cyto_scores, (-1, 1))), axis=1)
    merged_scores = np.concat((np.reshape(dinocell_scores, (-1, 1)), np.reshape(cellpose_sam_scores, (-1, 1)), np.reshape(samcell_livecell_scores, (-1, 1))), axis=1)
    rankings = np.argsort(np.argsort(-merged_scores)) + 1
    stds = np.std(merged_scores, axis=1)

    return rankings, stds

In [ ]:
mma_rankings, mma_stds = compute_rankings(dinocell_mmas, cellpose_sam_mmas, samcell_livecell_mmas, samcell_cyto_mmas)
mma_greedy_rankings, mma_greedy_stds = compute_rankings(dinocell_mma_greedys, cellpose_sam_mma_greedys, samcell_livecell_mma_greedys, samcell_cyto_mma_greedys)
aji_rankings, aji_stds = compute_rankings(dinocell_ajis, cellpose_sam_ajis, samcell_livecell_ajis, samcell_cyto_ajis)
seg_rankings, seg_stds = compute_rankings(dinocell_segs, cellpose_sam_segs, samcell_livecell_segs, samcell_cyto_segs)
pq_rankings, pq_stds = compute_rankings(dinocell_pqs, cellpose_sam_pqs, samcell_livecell_pqs, samcell_cyto_pqs)
ap_50_rankings, ap_50_stds = compute_rankings(dinocell_ap_50s, cellpose_sam_ap_50s, samcell_livecell_ap_50s, samcell_cyto_ap_50s)

# Disagreement Scores

In [ ]:
import itertools

def compute_metric_disagreement_with_mma(mma_rankings, metric_rankings):
    total_comparisons = 0
    disagreements = 0

    for i in range(len(mma_rankings)):
        curr_mma_rankings = mma_rankings[i]
        curr_metric_rankings = metric_rankings[i]

        index_sets = list(itertools.combinations(np.arange(len(mma_rankings[0])), 2))
        for index_set in index_sets:
            mma_0_rank = curr_mma_rankings[index_set[0]]
            mma_1_rank = curr_mma_rankings[index_set[1]]

            metric_0_rank = curr_metric_rankings[index_set[0]]
            metric_1_rank = curr_metric_rankings[index_set[1]]

            if mma_0_rank > mma_1_rank and metric_0_rank < metric_1_rank:
                disagreements += 1
            elif mma_0_rank < mma_1_rank and metric_0_rank > metric_1_rank:
                disagreements += 1

            total_comparisons += 1
    
    disagreement_rate = disagreements / total_comparisons

    return disagreement_rate

        

In [ ]:
def compute_top_1_disagreement_with_mma(mma_rankings, metric_rankings):
    mma_top_index = np.argmin(mma_rankings, axis=1)
    metric_top_index = np.argmin(metric_rankings, axis=1)

    top_1_matches = np.where(mma_top_index != metric_top_index, 1, 0)
    return np.mean(top_1_matches)

In [ ]:
aji_disagreement = compute_metric_disagreement_with_mma(mma_rankings, aji_rankings)
seg_disagreement = compute_metric_disagreement_with_mma(mma_rankings, seg_rankings)
pq_disagreement = compute_metric_disagreement_with_mma(mma_rankings, pq_rankings)
ap_50_disagreement = compute_metric_disagreement_with_mma(mma_rankings, ap_50_rankings)

In [ ]:
aji_top_1_disagreement = compute_top_1_disagreement_with_mma(mma_rankings, aji_rankings)
seg_top_1_disagreement = compute_top_1_disagreement_with_mma(seg_rankings, aji_rankings)
pq_top_1_disagreement = compute_top_1_disagreement_with_mma(pq_rankings, aji_rankings)
ap_50_top_1_disagreement = compute_top_1_disagreement_with_mma(ap_50_rankings, aji_rankings)

In [ ]:
aji_metrics = [round(aji_disagreement, 2), round(float(aji_top_1_disagreement), 2)]
seg_metrics = [round(seg_disagreement, 2), round(float(seg_top_1_disagreement), 2)]
pq_metrics = [round(pq_disagreement, 2), round(float(pq_top_1_disagreement), 2)]
ap_50_metrics = [round(ap_50_disagreement, 2), round(float(ap_50_top_1_disagreement), 2)]
print('      Disagreement   Top-1 Disagreement')
print(f'AJI:       {str(aji_metrics).replace(', ', '          ').replace('[', '').replace(']', '')}')
print(f'SEG:       {str(seg_metrics).replace(', ', '          ').replace('[', '').replace(']', '')}')
print(f'PQ:        {str(pq_metrics).replace(', ', '          ').replace('[', '').replace(']', '')}')
print(f'AP@50:     {str(ap_50_metrics).replace(', ', '          ').replace('[', '').replace(']', '')}')